# PART 04 · 5장 로지스틱 회귀분석 — 연습 문제 정답

교재 399쪽 연습 문제의 정답 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 공통 준비

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

path = 'data/part04_type3/chisquare_device_plan/device_plan.csv'
df = pd.read_csv(path)
print(df.shape)
print(df.head())

(420, 3)
  member_id device_type plan_grade
0     M0001      mobile    premium
1     M0002      mobile   standard
2     M0003      mobile   standard
3     M0004      mobile   standard
4     M0005      mobile    premium


## 문제 01

device_plan 데이터에서 plan_grade가 premium이면 1, 아니면 0인 premium_plan을 만든다. desktop을 기준 범주로 두고 device_type을 더미 변수로 변환한 뒤 로지스틱 회귀모형을 적합하여 다음 값을 구하시오.

- ① 유의수준 0.05에서 유의한 기기 유형 더미 변수
- ② tablet의 프리미엄 요금제 이용 오즈를 기준 범주와 비교한 오즈비

**정답 코드**

In [ ]:
df['premium_plan'] = (df['plan_grade'] == 'premium').astype(int)

device_dummy = pd.get_dummies(
    df['device_type'],
    prefix='device_type',
    dtype=int
).drop(columns=['device_type_desktop'])

X = sm.add_constant(device_dummy)
y = df['premium_plan']
model = sm.Logit(y, X).fit(disp=False)

alpha = 0.05
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)
})
print(result_table.round(4))

sig_vars = model.pvalues.drop('const')
sig_vars = sig_vars[sig_vars < alpha].index.tolist()
print('significant_variables:', sig_vars)
print(
    'tablet_odds_ratio:',
    round(float(np.exp(model.params['device_type_tablet'])), 4)
)

                      coef  p_value  odds_ratio
const              -0.6337   0.0002      0.5306
device_type_mobile -0.8978   0.0005      0.4075
device_type_tablet -1.1451   0.0009      0.3182
significant_variables: ['device_type_mobile', 'device_type_tablet']
tablet_odds_ratio: 0.3182


**간단 해설**

desktop을 기준 범주로 지정했다. mobile과 tablet 더미는 모두 유의수준 0.05에서 유의하다. tablet의 오즈비 0.3182는 다른 조건이 같을 때 tablet의 프리미엄 요금제 이용 오즈가 desktop의 약 0.3182배라는 뜻이다. 정답은 device_type_mobile, device_type_tablet과 0.3182다.

---

**정답** ① device_type_mobile, device_type_tablet, ② 0.3182